In [ ]:
# Installing dependencies
!pip install -q google-api-python-client youtube-transcript-api tqdm
print('All packages installed.')

All packages installed.


In [ ]:
# Configuration
import os
import re
import json
import time
import pandas as pd
from datetime import datetime, timedelta
from googleapiclient.errors import HttpError
from youtube_transcript_api import YouTubeTranscriptApi
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

PERSON  = 1  # change to 2 or 3 (we split up in order to use the YouTube API more effectively given restrictions)
API_KEY = "[redacted]"

BASE_DIR        = f"/content/drive/MyDrive/youtube_hype_P{PERSON}"
OUTPUT_FILE     = os.path.join(BASE_DIR, f"youtube_hype_P{PERSON}.csv")
CHECKPOINT_FILE = os.path.join(BASE_DIR, f"checkpoint_P{PERSON}.json")
os.makedirs(BASE_DIR, exist_ok=True)

PERSON_TECHNOLOGIES = {
    1: {
        "ChatGPT": {"launch": "2022-11-30", "query": "ChatGPT"},
        "GPT-4":   {"launch": "2023-03-14", "query": "GPT-4"},
    },
    2: {
        "Gemini":         {"launch": "2023-12-06", "query": "Gemini AI Google Gemini"},
        "GitHub_Copilot": {"launch": "2021-06-29", "query": "GitHub Copilot"},
    },
    3: {
        "SoraAI": {"launch": "2024-02-15", "query": "Sora AI OpenAI Sora"},
    },
}

TECHNOLOGIES         = PERSON_TECHNOLOGIES[PERSON]
VIDEOS_PER_WINDOW    = 15
MIN_VIEWS            = 1000
MIN_DURATION_SECONDS = 120
QUOTA_LIMIT          = 9000

from googleapiclient.discovery import build
youtube = build("youtube", "v3", developerKey=API_KEY)

print(f"Config loaded — Person {PERSON}")
print(f" Technologies : {list(TECHNOLOGIES.keys())}")
print(f" Output       : {OUTPUT_FILE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config loaded — Person 1
 Technologies : ['ChatGPT', 'GPT-4']
 Output       : /content/drive/MyDrive/youtube_hype_P1/youtube_hype_P1.csv


In [ ]:
# Helper function to assign primary technology of a video (with keywords and synonyms)
TECH_KEYWORDS = {
    "ChatGPT":        ["chatgpt", "chat gpt", "gpt-3.5", "gpt 3.5", "gpt3", "gpt 3"],
    "GPT-4":          ["gpt-4", "gpt4", "gpt 4", "gpt-4o", "gpt4o", "gpt-4 turbo", "gpt-4v"],
    "Gemini":         ["gemini", "google gemini", "gemini pro", "gemini ultra", "bard gemini", "google bard"],
    "GitHub_Copilot": ["github copilot", "copilot x", "copilot chat", "vs code copilot", "vscode copilot"],
    "SoraAI":         ["sora", "openai sora", "sora ai", "sora video", "sora model"],
}

def assign_primary_technology(title, description, transcript):
    """
    Count keyword hits weighted by field importance:
      title (3x) — most deliberate signal
      description (2x) — author-written context
      transcript (1x) — spoken content, noisier, varies by length
    Returns the technology with the highest score.
    """
    t = (title       or "").lower()
    d = (description or "").lower()
    r = (transcript  or "").lower()

    scores = {}
    for tech, keywords in TECH_KEYWORDS.items():
        scores[tech] = sum(
            t.count(kw) * 3 + d.count(kw) * 2 + r.count(kw)
            for kw in keywords
        )
    return max(scores, key=scores.get), scores

In [ ]:
# Helper functions to track API quota and save progress
quota_used = 0

def charge_quota(units):
    """Add units to daily counter; raise stop if limit reached."""
    global quota_used
    quota_used += units
    if quota_used >= QUOTA_LIMIT:
        raise RuntimeError(
            f"QUOTA_SAFE_STOP: {quota_used} units used today. "
            "Come back tomorrow and re-run."
        )

# Checkpoint helpers (saving progress so do not have to restart API calls if limit reached)
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)
    return {"tech_index": 0, "window_index": 0}

def save_checkpoint(tech_index, window_index):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"tech_index": tech_index, "window_index": window_index}, f)

def load_existing_rows():
    if os.path.exists(OUTPUT_FILE):
        df = pd.read_csv(OUTPUT_FILE)
        print(f"Loaded {len(df)} existing rows from {OUTPUT_FILE}")
        return df.to_dict("records")
    return []

def save_rows(rows):
    pd.DataFrame(rows).to_csv(OUTPUT_FILE, index=False)

In [ ]:
# Helper functions to get video data (search, get stats, get transcripts)

def generate_windows(launch_date_str):
    """Generate time windows for queries - 39 bi-weekly windows: 13 pre-launch + 26 post-launch."""
    launch = datetime.strptime(launch_date_str, "%Y-%m-%d")
    start  = launch - timedelta(weeks=26)
    windows = []
    for i in range(39):
        w_start = start + timedelta(weeks=2 * i)
        w_end   = w_start + timedelta(weeks=2)
        windows.append({
            "window_number":     i + 1,
            "window_start_date": w_start,
            "window_end_date":   w_end,
            "days_since_launch": (w_start - launch).days,
        })
    return windows

def parse_duration(iso):
    """Parse duration of videos"""
    if not iso:
        return None
    m = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", iso)
    if not m:
        return None
    h, mi, s = (int(x or 0) for x in m.groups())
    return h * 3600 + mi * 60 + s

def search_videos(query, published_after, published_before, max_results=80):
    """Wrapper for Youtube API search"""
    video_ids = []
    next_page = None
    while len(video_ids) < max_results:
        charge_quota(100)   # search.list = 100 units
        try:
            resp = youtube.search().list(
                part="id",
                q=query,
                type="video",
                publishedAfter=published_after.strftime("%Y-%m-%dT%H:%M:%SZ"),
                publishedBefore=published_before.strftime("%Y-%m-%dT%H:%M:%SZ"),
                # videoCaption="closedCaption",
                # videoDuration="medium",
                videoDuration="any",
                relevanceLanguage="en",
                maxResults=50,
                order="viewCount",
                pageToken=next_page,
            ).execute()
        except HttpError as e:
            _handle_http_error(e, context="search")
            return video_ids   # return what we have so far
        video_ids += [item["id"]["videoId"] for item in resp.get("items", [])]
        next_page = resp.get("nextPageToken")
        if not next_page:
            break
    return list(set(video_ids))

def get_video_details(video_ids):
    rows = []
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]
        charge_quota(len(batch))   # videos.list = 1 unit/video
        try:
            resp = youtube.videos().list(
                part="snippet,statistics,contentDetails",
                id=",".join(batch)
            ).execute()
        except HttpError as e:
            _handle_http_error(e, context="video details")
            continue
        for item in resp.get("items", []):
            snip  = item["snippet"]
            stats = item.get("statistics", {})
            cd    = item.get("contentDetails", {})
            rows.append({
                "video_id":        item["id"],
                "channel_id":      snip.get("channelId"),
                "channel_title":   snip.get("channelTitle"),
                "title":           snip.get("title"),
                "defaultLanguage":      snip.get("defaultLanguage"),
                "defaultAudioLanguage": snip.get("defaultAudioLanguage"),
                "description":     snip.get("description"),
                "tags":            snip.get("tags", []),
                "categoryId":      snip.get("categoryId"),
                "madeForKids":     snip.get("madeForKids", False),
                "published_at":    snip.get("publishedAt"),
                "duration_raw":    cd.get("duration"),
                "view_count":      int(stats.get("viewCount",  0)),
                "like_count":      int(stats.get("likeCount",  0)) if "likeCount"    in stats else None,
                "comment_count":   int(stats.get("commentCount",0)) if "commentCount" in stats else None
            })
    return rows

def get_channel_stats(channel_ids):
    out = {}
    for i in range(0, len(set(channel_ids)), 50):
        batch = list(set(channel_ids))[i:i+50]
        charge_quota(len(batch))
        try:
            resp = youtube.channels().list(
                part="statistics", id=",".join(batch)
            ).execute()
        except HttpError as e:
            _handle_http_error(e, context="channel stats")
            continue
        for item in resp.get("items", []):
            s = item.get("statistics", {})
            out[item["id"]] = {
                "channel_subscriber_count": int(s["subscriberCount"]) if "subscriberCount" in s else None,
                "channel_total_views":      int(s.get("viewCount", 0)),
                "channel_video_count":      int(s.get("videoCount", 0)),
            }
    return out

def get_transcript(video_id):
    try:
        ytt = YouTubeTranscriptApi()
        transcript_list = ytt.list(video_id)
        transcript = transcript_list.find_transcript(["en", "en-US", "en-GB"])
        segs = transcript.fetch()
        return " ".join(s.text for s in segs)
    except Exception:
        try:
            # fallback: auto-generated captions if no transcript
            ytt = YouTubeTranscriptApi()
            transcript_list = ytt.list(video_id)
            transcript = transcript_list.find_generated_transcript(["en", "en-US", "en-GB"])
            segs = transcript.fetch()
            return " ".join(s.text for s in segs)
        except Exception:
            return None

def _handle_http_error(e, context=""):
    """Re-raise quota errors; print + continue for everything else."""
    status = e.resp.status
    body   = str(e.content)
    if status == 403 and ("quotaExceeded" in body or "dailyLimitExceeded" in body):
        raise RuntimeError(
            f"QUOTA_SAFE_STOP: YouTube quota exhausted during [{context}]. "
            "Come back tomorrow and re-run Cell 4."
        )
    print(f"  HTTP {status} in [{context}] — skipping. Detail: {body[:120]}")

print("All helper functions defined.")

All helper functions defined.


In [ ]:
# Run / Resume data collection
quota_used = 0
all_rows   = load_existing_rows()
checkpoint = load_checkpoint()
tech_list  = list(TECHNOLOGIES.items())

print(f"\nResuming from tech_index={checkpoint['tech_index']}, window_index={checkpoint['window_index']}")
print(f"Quota limit: {QUOTA_LIMIT}")

try:
    for tech_index, (tech_name, config) in enumerate(tech_list):

        if tech_index < checkpoint["tech_index"]:
            print(f"  Skipping {tech_name}")
            continue

        print(f"\n{'='*55}\n  {tech_name}\n{'='*55}")

        windows = generate_windows(config["launch"])
        start_window = checkpoint["window_index"] if tech_index == checkpoint["tech_index"] else 0

        for win_index, win in enumerate(tqdm(windows, desc=tech_name, initial=start_window, total=39)):
            if win_index < start_window:
                continue

            video_ids = search_videos(
                query=config["query"],
                published_after=win["window_start_date"],
                published_before=win["window_end_date"],
            )
            if not video_ids:
                save_checkpoint(tech_index, win_index + 1)
                continue

            details = get_video_details(video_ids)

            details_parsed = []
            for d in details:
                d["duration_seconds"] = parse_duration(d.pop("duration_raw", None))
                details_parsed.append(d)

            details = [
                d for d in details_parsed
                if d["view_count"] >= MIN_VIEWS
                and (d["duration_seconds"] or 0) >= MIN_DURATION_SECONDS
                and (d.get("defaultAudioLanguage") or d.get("defaultLanguage") or "en").startswith("en")
            ]
            if not details:
                save_checkpoint(tech_index, win_index + 1)
                continue

            details.sort(key=lambda x: x["view_count"], reverse=True)
            details = details[:VIDEOS_PER_WINDOW]

            ch_ids   = [d["channel_id"] for d in details if d["channel_id"]]
            ch_stats = get_channel_stats(ch_ids) if ch_ids else {}

            for d in details:
                transcript = get_transcript(d["video_id"])
                cs         = ch_stats.get(d["channel_id"], {})

                primary_tech, tech_scores = assign_primary_technology(
                    d["title"], d["description"], transcript
                )

                if primary_tech != tech_name:
                    if tech_scores[primary_tech] > 0:
                        print(f"  Discarded '{d['title'][:60]}' → {primary_tech}")
                        continue

                row = {
                    "technology":            tech_name,
                    "tech_scores":           json.dumps(tech_scores),
                    **win,
                    **d,
                    **cs,
                    "transcript_text":       transcript,
                    "transcript_word_count": len(transcript.split()) if transcript else None,
                }
                all_rows.append(row)

            save_rows(all_rows)
            save_checkpoint(tech_index, win_index + 1)
            print(f"  window {win_index+1}/39 | {len(details)} videos | quota: {quota_used} | rows: {len(all_rows)}")
            time.sleep(0.5)

        print(f"\n  {tech_name} complete")
        save_checkpoint(tech_index + 1, 0)

    print(f"\nCOLLECTION COMPLETE — {len(all_rows)} rows")
    print(f"  Saved to: {OUTPUT_FILE}")

except RuntimeError as e:
    msg = str(e)
    if "QUOTA" in msg:
        print(f"\n{msg}")
        print(f"  Progress saved — {len(all_rows)} rows")
        print(f"  Checkpoint: {load_checkpoint()}")
        print(f"  Re-run Cell 4 tomorrow.")
    else:
        save_rows(all_rows)
        raise

except Exception as e:
    save_rows(all_rows)
    print(f"\nUnexpected error: {e}")
    raise

Loaded 632 existing rows from /content/drive/MyDrive/youtube_hype_P1/youtube_hype_P1.csv

Resuming from tech_index=2, window_index=0
Quota limit: 9000
  Skipping ChatGPT
  Skipping GPT-4

COLLECTION COMPLETE — 632 rows
  Saved to: /content/drive/MyDrive/youtube_hype_P1/youtube_hype_P1.csv
